# 04_feature_fusion

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA

In [13]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FUSION_DIR = DATA_DIR / "fusion"

FUSION_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(DATA_DIR)

d:\360Downloads\bioinformatics\Task\AIP
d:\360Downloads\bioinformatics\Task\AIP\data\processed


In [14]:
# 加载3类特征

# Handcrafted
X_train_hand = np.load(DATA_DIR / "handcrafted" / "X_train_handcrafted.npy")
X_test_hand = np.load(DATA_DIR / "handcrafted" / "X_test_handcrafted.npy")
y_train = np.load(DATA_DIR / "handcrafted" / "y_train.npy")
y_test = np.load(DATA_DIR / "handcrafted" / "y_test.npy")

# ProtT5
X_train_prott5 = np.load(DATA_DIR / "prott5" / "X_train_prott5.npy")
X_test_prott5 = np.load(DATA_DIR / "prott5" / "X_test_prott5.npy")

# ESM2
X_train_esm2 = np.load(DATA_DIR / "esm2" / "X_train_esm2.npy")
X_test_esm2 = np.load(DATA_DIR / "esm2" / "X_test_esm2.npy")


In [15]:
feature_names_hand = pd.read_csv(
    DATA_DIR / "handcrafted" / "handcrafted_feature_names.csv"
)["feature_name"].tolist()

In [17]:
# 检查维度
print("Handcrafted:", X_train_hand.shape)
print("ESM2:", X_train_esm2.shape)
print("ProtT5:", X_train_prott5.shape)

Handcrafted: (3583, 441)
ESM2: (3583, 480)
ProtT5: (3583, 768)


## 1.手工特征

In [18]:
# 手工特征筛选：
# （1）筛选函数
def filter_handcrafted_features(
    X_train,
    X_test,
    y_train,
    feature_names,
    variance_threshold=0.0,
    corr_threshold=0.95,
    k_best=50
):
    """
    对手工特征做三级筛选：
    1. 低方差过滤
    2. 高相关过滤
    3. 基于互信息的 SelectKBest

    返回：
    X_train_selected, X_test_selected, selected_feature_names
    """
    # -------------------------
    # Step 1: 方差过滤
    # -------------------------
    vt = VarianceThreshold(threshold=variance_threshold)
    X_train_v = vt.fit_transform(X_train)
    X_test_v = vt.transform(X_test)

    names_v = [name for name, keep in zip(feature_names, vt.get_support()) if keep]

    print(f"方差过滤前: {X_train.shape[1]} 维")
    print(f"方差过滤后: {X_train_v.shape[1]} 维")

    # -------------------------
    # Step 2: 高相关过滤
    # -------------------------
    df_train_v = pd.DataFrame(X_train_v, columns=names_v)
    corr_matrix = df_train_v.corr().abs()

    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]

    names_c = [name for name in names_v if name not in to_drop]
    keep_idx = [names_v.index(name) for name in names_c]

    X_train_c = X_train_v[:, keep_idx]
    X_test_c = X_test_v[:, keep_idx]

    print(f"相关性过滤后: {X_train_c.shape[1]} 维")
    print(f"删除高相关特征数: {len(to_drop)}")

    # -------------------------
    # Step 3: SelectKBest
    # -------------------------
    k_best = min(k_best, X_train_c.shape[1])

    skb = SelectKBest(score_func=mutual_info_classif, k=k_best)
    X_train_s = skb.fit_transform(X_train_c, y_train)
    X_test_s = skb.transform(X_test_c)

    names_s = [name for name, keep in zip(names_c, skb.get_support()) if keep]

    print(f"最终筛选后: {X_train_s.shape[1]} 维")

    return X_train_s, X_test_s, names_s

In [19]:
X_train_hand_sel, X_test_hand_sel, selected_hand_features = filter_handcrafted_features(
    X_train=X_train_hand,
    X_test=X_test_hand,
    y_train=y_train,
    feature_names=feature_names_hand,
    variance_threshold=0.0,
    corr_threshold=0.95,
    k_best=50
)

print("原始 handcrafted:", X_train_hand.shape)
print("筛选后 handcrafted:", X_train_hand_sel.shape)
print("前10个保留特征：")
print(selected_hand_features[:10])

方差过滤前: 441 维
方差过滤后: 441 维
相关性过滤后: 437 维
删除高相关特征数: 4
最终筛选后: 50 维
原始 handcrafted: (3583, 441)
筛选后 handcrafted: (3583, 50)
前10个保留特征：
['AAC_A', 'AAC_C', 'AAC_D', 'AAC_E', 'AAC_F', 'AAC_G', 'AAC_H', 'AAC_I', 'AAC_K', 'AAC_L']


In [ ]:
# 保存筛选过后的手工特征
hand_sel_dir = FUSION_DIR / "handcrafted_selected"
hand_sel_dir.mkdir(parents=True, exist_ok=True)

np.save(hand_sel_dir / "X_train.npy", X_train_hand_sel)
np.save(hand_sel_dir / "X_test.npy", X_test_hand_sel)
np.save(hand_sel_dir / "y_train.npy", y_train)
np.save(hand_sel_dir / "y_test.npy", y_test)

pd.DataFrame({"feature_name": selected_hand_features}).to_csv(
    hand_sel_dir / "selected_feature_names.csv",
    index=False
)

## 2.开始构造融合特征

In [22]:
def concatenate_features(*arrays):
    return np.concatenate(arrays, axis=1)

### （1）Handcrafted + ProtT5

In [23]:
X_train_hand_prott5 = concatenate_features(X_train_hand_sel, X_train_prott5)
X_test_hand_prott5 = concatenate_features(X_test_hand_sel, X_test_prott5)

print("handcrafted + prott5:")
print(X_train_hand_prott5.shape, X_test_hand_prott5.shape)

handcrafted + prott5:
(3583, 818) (897, 818)


### （2）Handcrafted + ESM2

In [24]:
X_train_hand_esm2 = concatenate_features(X_train_hand_sel, X_train_esm2)
X_test_hand_esm2 = concatenate_features(X_test_hand_sel, X_test_esm2)

print("handcrafted + esm2:")
print(X_train_hand_esm2.shape, X_test_hand_esm2.shape)

handcrafted + esm2:
(3583, 530) (897, 530)


### （3）ProtT5 + ESM2

In [25]:
X_train_prott5_esm2 = concatenate_features(X_train_prott5, X_train_esm2)
X_test_prott5_esm2 = concatenate_features(X_test_prott5, X_test_esm2)

print("prott5 + esm2:")
print(X_train_prott5_esm2.shape, X_test_prott5_esm2.shape)

prott5 + esm2:
(3583, 1248) (897, 1248)


### （4）三者全融合

In [26]:
X_train_all = concatenate_features(X_train_hand_sel, X_train_prott5, X_train_esm2)
X_test_all = concatenate_features(X_test_hand_sel, X_test_prott5, X_test_esm2)

print("all fusion:")
print(X_train_all.shape, X_test_all.shape)

all fusion:
(3583, 1298) (897, 1298)


## 3.PCA降维

In [36]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def apply_pca(X_train, X_test, n_components=0.95):
    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # PCA
    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    print("原始维度:", X_train.shape[1])
    print("PCA后维度:", X_train_pca.shape[1])
    print("累计解释方差:", pca.explained_variance_ratio_.sum())

    return X_train_pca, X_test_pca, pca, scaler

### （1）Handcrafted + ProtT5 + PCA

In [39]:
X_train_hand_prott5_pca, X_test_hand_prott5_pca, pca_hand_prott5, scaler_hand_prott5 = apply_pca(
    X_train_hand_prott5,
    X_test_hand_prott5,
    n_components=0.95
)

原始维度: 818
PCA后维度: 8
累计解释方差: 0.9508172954827103


### （2）Handcrafted + ESM2 + PCA

In [40]:
X_train_hand_esm2_pca, X_test_hand_esm2_pca, pca_hand_esm2,scaler_hand_esm2 = apply_pca(
    X_train_hand_esm2,
    X_test_hand_esm2,
    n_components=0.95
)

原始维度: 530
PCA后维度: 86
累计解释方差: 0.950675567961063


### （3）ProtT5 + ESM2 + PCA

In [41]:
X_train_prott5_esm2_pca, X_test_prott5_esm2_pca, pca_prott5_esm2, scaler_prott5_esm2 = apply_pca(
    X_train_prott5_esm2,
    X_test_prott5_esm2,
    n_components=0.95
)

原始维度: 1248
PCA后维度: 39
累计解释方差: 0.95059824


### （4）三者全融合 + PCA

In [42]:
X_train_all_pca, X_test_all_pca, pca_all, scaler_all = apply_pca(
    X_train_all,
    X_test_all,
    n_components=0.95
)

原始维度: 1298
PCA后维度: 46
累计解释方差: 0.9505883685321349


In [43]:
# 保存非PCA版本和PCA版本

def save_feature_version(save_dir, X_train, X_test, y_train, y_test):
    save_dir.mkdir(parents=True, exist_ok=True)
    np.save(save_dir / "X_train.npy", X_train)
    np.save(save_dir / "X_test.npy", X_test)
    np.save(save_dir / "y_train.npy", y_train)
    np.save(save_dir / "y_test.npy", y_test)

save_feature_version(FUSION_DIR / "handcrafted_prott5", X_train_hand_prott5, X_test_hand_prott5, y_train, y_test)
save_feature_version(FUSION_DIR / "handcrafted_esm2", X_train_hand_esm2, X_test_hand_esm2, y_train, y_test)
save_feature_version(FUSION_DIR / "prott5_esm2", X_train_prott5_esm2, X_test_prott5_esm2, y_train, y_test)
save_feature_version(FUSION_DIR / "all_fusion", X_train_all, X_test_all, y_train, y_test)

save_feature_version(FUSION_DIR / "handcrafted_prott5_pca", X_train_hand_prott5_pca, X_test_hand_prott5_pca, y_train, y_test)
save_feature_version(FUSION_DIR / "handcrafted_esm2_pca", X_train_hand_esm2_pca, X_test_hand_esm2_pca, y_train, y_test)
save_feature_version(FUSION_DIR / "prott5_esm2_pca", X_train_prott5_esm2_pca, X_test_prott5_esm2_pca, y_train, y_test)
save_feature_version(FUSION_DIR / "all_fusion_pca", X_train_all_pca, X_test_all_pca, y_train, y_test)

In [44]:
summary_df = pd.DataFrame([
    ["handcrafted_selected", X_train_hand_sel.shape[1], X_test_hand_sel.shape[1]],
    ["handcrafted_prott5", X_train_hand_prott5.shape[1], X_test_hand_prott5.shape[1]],
    ["handcrafted_esm2", X_train_hand_esm2.shape[1], X_test_hand_esm2.shape[1]],
    ["prott5_esm2", X_train_prott5_esm2.shape[1], X_test_prott5_esm2.shape[1]],
    ["all_fusion", X_train_all.shape[1], X_test_all.shape[1]],
    ["handcrafted_prott5_pca", X_train_hand_prott5_pca.shape[1], X_test_hand_prott5_pca.shape[1]],
    ["handcrafted_esm2_pca", X_train_hand_esm2_pca.shape[1], X_test_hand_esm2_pca.shape[1]],
    ["prott5_esm2_pca", X_train_prott5_esm2_pca.shape[1], X_test_prott5_esm2_pca.shape[1]],
    ["all_fusion_pca", X_train_all_pca.shape[1], X_test_all_pca.shape[1]],
], columns=["feature_set", "train_dim", "test_dim"])

summary_df

,feature_set,train_dim,test_dim
0,handcrafted_selected,50,50
1,handcrafted_prott5,818,818
2,handcrafted_esm2,530,530
3,prott5_esm2,1248,1248
4,all_fusion,1298,1298
5,handcrafted_prott5_pca,8,8
6,handcrafted_esm2_pca,86,86
7,prott5_esm2_pca,39,39
8,all_fusion_pca,46,46
